In [1]:
import pyomo.environ as pyo

# Item catalog: each entry has a value (utility we get from packing it)
# and a weight (capacity it consumes in the knapsack).
data = {
    'laptop': {
        'value': 25,
        'weight': 6
    },
    'water_bottle': {
        'value': 4,
        'weight': 2
    },
    'tent': {
        'value': 18,
        'weight': 9
    },
    'sleeping_bag': {
        'value': 14,
        'weight': 5
    },
    'flashlight': {
        'value': 6,
        'weight': 1
    },
    'first_aid_kit': {
        'value': 10,
        'weight': 3
    },
    'stove': {
        'value': 12,
        'weight': 4
    },
    'jacket': {
        'value': 11,
        'weight': 4
    },
    'map': {
        'value': 5,
        'weight': 1
    },
    'camera': {
        'value': 16,
        'weight': 3
    }
}

# Total weight capacity of the knapsack.
weight_limit = 14

# Build a concrete Pyomo model (all data known up front, no abstract symbols).
m = pyo.ConcreteModel()

# Index set: the names of the candidate items.
m.things = pyo.Set(initialize=data.keys())

# Decision variable: y[i] = 1 if item i is packed, 0 otherwise (binary => 0/1 only).
m.y = pyo.Var(m.things, within=pyo.Binary)

# Objective: maximize the total value of the items selected.
m.value = pyo.Objective(
    expr=sum(data[i]['value'] * m.y[i] for i in m.things),
    sense=pyo.maximize
)

# Capacity constraint: total weight of chosen items must not exceed the limit.
m.weight = pyo.Constraint(
    expr=sum(data[i]['weight'] * m.y[i] for i in m.things) <= weight_limit
)

In [2]:
# Use HiGHS as the MILP solver (ships as a pip wheel via highspy).
solver = pyo.SolverFactory('appsi_highs')

# Solve the model; tee=True streams the solver's log to stdout.
results = solver.solve(m, tee=True)

print("\nOptimal things:")

# Print every item the solver picked. The 0.5 threshold guards against
# tiny floating-point noise around an integer 0/1 value.
for i in m.things:
    if pyo.value(m.y[i]) > 0.5:
        print(i)

# Optimal objective value (sum of values of chosen items).
print("\nTotal value =", pyo.value(m.value))

# Sanity check: total weight of the chosen items should be <= weight_limit.
print(
    "Total weight =",
    sum(data[i]['weight'] * pyo.value(m.y[i]) for i in m.things)
)

Running HiGHS 1.14.0 (git hash: 7df0786): Copyright (c) 2026 under MIT licence terms
MIP has 1 row; 10 cols; 10 nonzeros; 10 integer variables (10 binary)
Coefficient ranges:
  Matrix  [1e+00, 9e+00]
  Cost    [4e+00, 2e+01]
  Bound   [1e+00, 1e+00]
  RHS     [1e+01, 1e+01]
Presolving model
1 rows, 10 cols, 10 nonzeros 0s
1 rows, 9 cols, 9 nonzeros 0s
Presolve reductions: rows 1(-0); columns 9(-1); nonzeros 9(-1) 
Objective function is integral with scale 1

Solving MIP model with:
   1 row
   9 cols (9 binary, 0 integer, 0 implied int., 0 continuous, 0 domain fixed)
   9 nonzeros

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility jump; L => Sub-MIP; P => Empty MIP; R => Randomized rounding;
     S => Solve LP; T => Evaluate node; U => Unbounded; X => User solution; Y => HiGHS solution;
     Z => ZI Round; l => Trivial lower; p => Trivial point; u => Trivial upper; z => Trivial zero

        Nodes      |    B&B Tree 